# 04 · Etiquetas tardías

Ejecutar todas las celdas en orden. Datos históricos y simulaciones educativas; no representan una aplicación financiera real.

In [1]:
from pathlib import Path
import sys, os
os.environ["EVIDENTLY_DO_NOT_TRACK"]="1"
base=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"_config.yml").exists())
project=base/"actividad_3/proyecto_inicial"
sys.path.insert(0,str(project/"src"))
os.environ["BANK_ROOT"]=str(project)
import numpy as np, pandas as pd
from bank_ops.data import load, partitions
from bank_ops.config import FEATURES, SEED
from bank_ops.model import pipeline, metrics
data=load(); train,validation,test=partitions(data)


In [2]:
estimator=pipeline();estimator.fit(train[FEATURES],(train.y=="yes").astype(int))
from bank_ops.monitor import delayed_performance
batch=validation.iloc[:500]
pred=pd.DataFrame({"prediction_id":[f"demo-{i}" for i in batch.row_id],"probability":estimator.predict_proba(batch[FEATURES])[:,1],"model_version":"baseline-notebook"})
labels=pd.DataFrame({"prediction_id":pred.prediction_id.iloc[:300],"target":(batch.y.iloc[:300]=="yes").astype(int).to_numpy()})
display(pd.DataFrame(delayed_performance(pred,labels)))

,model_version,total,labeled,coverage,metrics
0,baseline-notebook,500,300,0.6,"{'n': 300, 'prevalence': 0.04666666666666667, ..."


## Maduración
Se simula llegada parcial. Las primeras etiquetas no representan necesariamente una muestra aleatoria.

In [3]:
complete=pd.DataFrame({"prediction_id":pred.prediction_id,"target":(batch.y=="yes").astype(int).to_numpy()})
display(pd.DataFrame(delayed_performance(pred,complete)))
try: delayed_performance(pred,pd.concat([labels,labels.iloc[:1]]))
except ValueError as error: print("Error esperado:",error)

,model_version,total,labeled,coverage,metrics
0,baseline-notebook,500,500,1.0,"{'n': 500, 'prevalence': 0.06, 'average_precis..."


Error esperado: Identificadores duplicados


## Reflexión
Comparar resultado parcial y maduro. Proponer política de espera y evitar doble conteo. No convertir resultados ausentes en negativos.